# CWT-Based Gaze Error Spectrogram Pipeline & Edge AI Engine

**Statistical Learning for AI Lab — VOG-MCI Detection System**

---

## Project Overview

This notebook implements a complete end-to-end pipeline for **binary classification of Mild Cognitive Impairment (MCI) vs. Healthy Controls (HC)** using Video-Oculography (VOG) saccade recordings.

The core premise is that MCI patients exhibit measurable degradation in oculomotor control — specifically, abnormal **gaze error dynamics** during visually-guided saccade tasks. By transforming the 1-D gaze error signal into the time-frequency domain via the Continuous Wavelet Transform (CWT), we obtain a 2-D scalogram that simultaneously encodes:
- **Reaction latency** (temporal axis): delayed saccade initiation
- **Micro-saccadic tremor** (frequency axis): high-frequency instability during fixation

### Pipeline Architecture

```
Raw VOG CSV
  │
  ▼
[Layer 1]  Event-Locked CWT Pipeline
           Trigger detection → Epoch extraction → Gaze error → Complex Morlet CWT
           Output: [2, 40, 100] tensor  (Real + Imag channels)
  │
  ├──────► [Layer 5]  XAI Visualizer
  │                   Group-mean dB scalograms → SPM-style Difference Map
  │
  ▼
[Layer 2]  VOG_CWT_Dataset
           Per-channel Z-score normalization, subject-ID tracking
  │
  ▼
[Layer 3]  EdgeCWTClassifier  (CNN–CBAM Hybrid)
           3× [DepthwiseSep-Conv → CBAM] → GAP → FC(64→32→2)
  │
  ▼
[Layer 4a] ModelTrainer
           Subject-level stratified split, Weighted CE Loss, CosineAnnealingLR
  │
  ▼
[Layer 4b] JetsonInferenceEngine
           Per-epoch inference → Soft-voting ensemble → MCI probability
```

### Dataset

| Group | Subjects (local) | CSV files | Label |
|-------|-----------------|-----------|-------|
| HC    | 14              | 116       | 0     |
| MCI   | 12              | 96        | 1     |
| MCI+  | 11              | 88        | 1     |
| **Total** | **37**      | **300**   | binary|

An additional cohort of comparable size (~37 subjects) is reserved for fine-tuning, yielding a total of approximately **74 subjects**.

In [ ]:
import os
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict, Counter
import pywt
from scipy.ndimage import zoom

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset

---

## Attention Module: CBAM (Convolutional Block Attention Module)

### Motivation

A plain CNN applies uniform weights across all channels and all spatial positions of a feature map. For CWT scalograms, however, the clinically informative signal is **sparse and localized** in time-frequency space — for example, MCI-related latency deficits concentrate around $t \in [0.2, 0.4]$ s post-stimulus, and tremor instability concentrates in $f \in [15, 30]$ Hz. CBAM provides a **data-driven gating mechanism** that suppresses uninformative regions and amplifies diagnostically relevant ones, directly aligning with the XAI objective of identifying time-frequency Regions of Interest (RoIs).

### Channel Attention — *What* to emphasize

Given a feature map $\mathbf{F} \in \mathbb{R}^{C \times H \times W}$, the channel attention map $\mathbf{M}_c \in \mathbb{R}^{C}$ is computed via a shared MLP over both average-pooled and max-pooled descriptors:

$$\mathbf{M}_c(\mathbf{F}) = \sigma\!\left(\mathrm{MLP}\bigl(\mathbf{F}^c_{\mathrm{avg}}\bigr) + \mathrm{MLP}\bigl(\mathbf{F}^c_{\mathrm{max}}\bigr)\right)$$

where $\mathbf{F}^c_{\mathrm{avg}} = \frac{1}{HW}\sum_{h,w} \mathbf{F}_{:,h,w}$ captures global statistics and $\mathbf{F}^c_{\mathrm{max}}$ captures salient activations. The shared MLP has a bottleneck of $\lfloor C/r \rfloor$ neurons ($r=4$ here).

### Spatial Attention — *Where* to look

The spatial attention map $\mathbf{M}_s \in \mathbb{R}^{1 \times H \times W}$ operates on channel-pooled representations:

$$\mathbf{M}_s(\mathbf{F}') = \sigma\!\left(f^{7\times7}\bigl([\mathbf{F}'^s_{\mathrm{avg}}\,;\,\mathbf{F}'^s_{\mathrm{max}}]\bigr)\right)$$

where $[\,;\,]$ denotes channel-wise concatenation and $f^{7\times7}$ is a $7\times7$ convolution. The $7\times7$ kernel spans a broad receptive field in the $(f, t)$ plane, appropriate for detecting distributed latency and frequency patterns.

The full CBAM refinement is applied sequentially: $\mathbf{F}'' = \mathbf{M}_s(\mathbf{F}') \otimes \mathbf{F}'$, where $\mathbf{F}' = \mathbf{M}_c(\mathbf{F}) \otimes \mathbf{F}$.

> **XAI connection:** The spatial attention weight matrix $\mathbf{M}_s$ at the final feature stage can be upsampled back to the original scalogram dimensions and overlaid as a learned saliency map — providing a direct, model-intrinsic explanation of *which time-frequency region drove the classification decision*.

In [ ]:
# =========================================================================
# CBAM: Convolutional Block Attention Module
# Channel attention (WHAT to focus on) + Spatial attention (WHERE to focus)
# Spatial attention aligns with XAI goal: highlight Time-Freq RoIs
# =========================================================================
class ChannelAttention(nn.Module):
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        mid = max(channels // reduction, 2)
        self.fc = nn.Sequential(
            nn.Linear(channels, mid, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c = x.size(0), x.size(1)
        avg = self.fc(self.avg_pool(x).view(b, c))
        mx  = self.fc(self.max_pool(x).view(b, c))
        return self.sigmoid(avg + mx).view(b, c, 1, 1) * x


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg    = torch.mean(x, dim=1, keepdim=True)
        mx, _  = torch.max(x,  dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg, mx], dim=1))) * x


class CBAM(nn.Module):
    def __init__(self, channels, reduction=4, spatial_kernel=7):
        super().__init__()
        self.ca = ChannelAttention(channels, reduction)
        self.sa = SpatialAttention(spatial_kernel)

    def forward(self, x):
        return self.sa(self.ca(x))

---

## Layer 1 — Event-Locked CWT Pipeline

### 1.1 Clinical Signal: Gaze Error During Saccades

A **saccade** is a rapid ballistic eye movement that re-foveates a newly appeared target. In MCI, documented oculomotor deficits include:
- **Increased latency** ($\Delta t \in [50, 200]$ ms): delayed initiation of the corrective saccade
- **Hypometria** (undershoot): the eye falls short of the target ($e > 0$)
- **Hypermetria** (overshoot): the eye overshoots the target ($e < 0$)
- **Increased post-saccadic oscillations**: high-frequency instability at fixation

The **signed gaze error** is defined as:

$$e(t) = \theta_{\mathrm{target}}(t) - \theta_{\mathrm{actual}}(t)$$

Note that the absolute value is deliberately *not* taken — preserving sign distinguishes hypometria ($e > 0$) from hypermetria ($e < 0$), which have different neural substrates and may carry distinct discriminative power for MCI.

### 1.2 Event-Locked Epoching

The stimulus onset time $t_k$ is detected as the first moment the target position changes:

$$t_k = \{t : \theta_{\mathrm{target}}(t) \neq \theta_{\mathrm{target}}(t - \Delta t)\}$$

Each epoch is extracted as a fixed window relative to $t_k$:

$$\mathcal{E}_k = e(t) \text{ for } t \in [t_k - 0.2\,\mathrm{s},\; t_k + 0.8\,\mathrm{s}]$$

**Baseline correction** removes the pre-stimulus DC offset to eliminate systematic fixation bias:

$$\tilde{e}_k(t) = e_k(t) - \underbrace{\frac{1}{N_{\mathrm{pre}}} \sum_{t < t_k} e_k(t)}_{\text{pre-stimulus mean}}$$

This is analogous to the demeaning step in ERP (Event-Related Potential) analysis and ensures the CWT captures *change from baseline* rather than absolute gaze position.

### 1.3 Continuous Wavelet Transform with Complex Morlet

The CWT of a signal $f(t)$ with respect to a mother wavelet $\psi$ is:

$$W_\psi[f](a, b) = \frac{1}{\sqrt{a}} \int_{-\infty}^{\infty} f(t)\, \overline{\psi\!\left(\frac{t - b}{a}\right)} \, dt$$

where $a > 0$ is the **scale** (inversely proportional to frequency) and $b$ is the **time shift**. The frequency-scale relationship is:

$$f = \frac{f_c}{a \cdot \Delta t}$$

where $f_c$ is the center frequency of $\psi$. We use the **Complex Morlet wavelet** (`cmor5.0-1.0`):

$$\psi_{\mathrm{cmor}}(t) = \frac{1}{\sqrt{\pi B}} e^{2\pi i f_c t} \, e^{-t^2 / B}$$

with bandwidth parameter $B = 1.0$ and $f_c = 5.0$ Hz. This wavelet provides a favorable trade-off between **time resolution** (important for latency estimation) and **frequency resolution** (important for tremor characterization), governed by the Heisenberg-Gabor uncertainty principle: $\sigma_t \cdot \sigma_f \geq \frac{1}{4\pi}$.

**Superiority over STFT:** The STFT uses a fixed window length, yielding uniform time-frequency resolution. The CWT adaptively uses short windows at high frequencies (good time resolution for fast events) and long windows at low frequencies (good frequency resolution for slow oscillations) — critical for capturing both micro-saccadic tremor and reaction latency in the same scalogram.

### 1.4 Two-Channel Complex Output

Rather than discarding phase information via $|W_\psi[\tilde{e}]|^2$, the pipeline retains the full complex output as two separate channels:

$$\mathbf{Z}_k = \begin{bmatrix} \mathrm{Re}(W_\psi[\tilde{e}_k]) \\ \mathrm{Im}(W_\psi[\tilde{e}_k]) \end{bmatrix} \in \mathbb{R}^{2 \times F \times T}$$

The real part encodes cosine-phase components (aligned with the positive gaze error direction, i.e., hypometria), while the imaginary part encodes sine-phase components. Together they allow the network to reconstruct both magnitude and instantaneous phase, providing richer discriminative features than power alone.

In [ ]:
# =========================================================================
# [Layer 1] Data Engineering: Event-Locked CWT Pipeline
# Enhancements:
#   - Logarithmic frequency spacing (finer resolution at low freqs)
#   - Binocular 4-channel output [Re_L, Im_L, Re_R, Im_R] per epoch
#   - Artifact rejection (peak gaze error > threshold → skip epoch)
#   - 3-level data_store: group → subject_id → task → [tensors]
# =========================================================================
class EventLockedCWTPipeline:
    def __init__(self, pre_stimulus_sec=0.2, post_stimulus_sec=0.8,
                 min_freq=1.0, max_freq=40.0, freq_bins=40,
                 target_time_bins=100, w_morlet=5.0,
                 artifact_threshold=30.0):
        self.pre_sec            = pre_stimulus_sec
        self.post_sec           = post_stimulus_sec
        self.min_freq           = min_freq
        self.max_freq           = max_freq
        self.freq_bins          = freq_bins
        self.target_time_bins   = target_time_bins
        self.w                  = w_morlet
        self.artifact_threshold = artifact_threshold
        # LOG spacing: neural oscillations follow 1/f → more bins at low freq
        self.frequencies = np.logspace(np.log10(min_freq), np.log10(max_freq), freq_bins)

        self.target_tasks = {
            "Horizontal": ["Horizontal Saccade A", "Horizontal Saccade B",
                           "Horizontal Saccade B (anti)", "Horizontal Saccade R"],
            "Vertical":   ["Vertical Saccade A",   "Vertical Saccade B",
                           "Vertical Saccade B (anti)",   "Vertical Saccade R"],
        }
        # 3-level store: group -> subject_id -> task -> [binocular tensors]
        self.data_store = defaultdict(
            lambda: defaultdict(
                lambda: defaultdict(list)
            )
        )

    # ------------------------------------------------------------------
    def _load_csv_safely(self, file_path: Path) -> pd.DataFrame:
        try:
            df = pd.read_csv(file_path, skipinitialspace=True)
            df.columns = [str(c).strip().lower() for c in df.columns]
            if any('lh' in c for c in df.columns):
                return df.apply(pd.to_numeric, errors='coerce').dropna(how='all').reset_index(drop=True)
        except Exception:
            pass
        for enc in ['utf-16', 'utf-16le', 'utf-8-sig', 'cp949']:
            try:
                with open(file_path, 'r', encoding=enc, errors='replace') as f:
                    lines = f.readlines()
                for i, line in enumerate(lines):
                    line_clean = line.replace('\x00', '').lower()
                    if 'lh' in line_clean and 'rh' in line_clean:
                        header_cols = [col.replace('\x00', '').strip().lower() for col in line.split(',')]
                        parsed = [
                            [v.strip() for v in l.replace('\x00', '').strip().split(',')]
                            for l in lines[i + 1:] if l.strip()
                        ]
                        df = pd.DataFrame(parsed, columns=header_cols)
                        return df.apply(pd.to_numeric, errors='coerce').dropna(how='all').reset_index(drop=True)
            except UnicodeError:
                continue
        raise ValueError(f"Headers missing or unreadable in {file_path.name}")

    # ------------------------------------------------------------------
    def _cwt_one_signal(self, signal, scales, wavelet_name, dt):
        """CWT for a single 1-D signal. Returns (real_resized, imag_resized)."""
        cwtm, _ = pywt.cwt(signal, scales, wavelet_name, sampling_period=dt)
        tz = self.target_time_bins / cwtm.shape[1]
        return (zoom(np.real(cwtm), (1.0, tz), mode='nearest', order=1),
                zoom(np.imag(cwtm), (1.0, tz), mode='nearest', order=1))

    # ------------------------------------------------------------------
    def _extract_binocular_epochs(self, df, target_col, left_col, right_col, fs):
        """
        Returns list of [4, freq, time] tensors: [Re_L, Im_L, Re_R, Im_R].
        Combines both eyes into one binocular epoch; skips artifacts.
        """
        target_val = df[target_col].fillna(0).values
        left_val   = df[left_col].fillna(0).values
        right_val  = df[right_col].fillna(0).values
        event_indices = np.where(np.diff(target_val, prepend=0) != 0)[0]

        samples_pre  = int(self.pre_sec  * fs)
        samples_post = int(self.post_sec * fs)
        dt           = 1.0 / fs
        wavelet_name = f'cmor{self.w}-1.0'
        scales       = pywt.central_frequency(wavelet_name) / (self.frequencies * dt)

        valid_cwts = []
        for idx in event_indices:
            s, e = idx - samples_pre, idx + samples_post
            if s < 0 or e > len(df):
                continue

            err_L = target_val[s:e] - left_val[s:e]
            err_L = err_L - np.mean(err_L[:samples_pre])
            err_R = target_val[s:e] - right_val[s:e]
            err_R = err_R - np.mean(err_R[:samples_pre])

            # Artifact rejection: skip if either eye exceeds threshold (degrees)
            if (np.max(np.abs(err_L)) > self.artifact_threshold or
                    np.max(np.abs(err_R)) > self.artifact_threshold):
                continue

            re_L, im_L = self._cwt_one_signal(err_L, scales, wavelet_name, dt)
            re_R, im_R = self._cwt_one_signal(err_R, scales, wavelet_name, dt)

            # Stack: [Re_L, Im_L, Re_R, Im_R] → [4, freq, time]
            valid_cwts.append(np.stack([re_L, im_L, re_R, im_R], axis=0))

        return valid_cwts

    # ------------------------------------------------------------------
    def process_directory(self, base_dir: Path):
        csv_files = [f for f in base_dir.rglob('*.csv') if 'PD VOG' in f.name]
        processed = 0
        for filepath in csv_files:
            clean_task = filepath.stem.replace("PD VOG -_", "").replace("PD VOG -", "").strip()
            axis_type = ("Horizontal" if "Horizontal" in clean_task
                         else "Vertical" if "Vertical" in clean_task else None)
            if not axis_type or clean_task not in self.target_tasks[axis_type]:
                continue

            group = None
            cur = filepath.parent
            while cur != base_dir and cur != cur.parent:
                if cur.name.upper().startswith("HC_csv_24_25"):  group = "HC_csv_24_25"; break
                if cur.name.upper().startswith("MCI"): group = "MCI"; break
                cur = cur.parent
            if not group:
                for part in filepath.parts:
                    if part.upper().startswith("HC_csv_24_25"):  group = "HC_csv_24_25"; break
                    if part.upper().startswith("MCI"): group = "MCI"; break
            if not group:
                continue

            subject_id = filepath.parent.name
            try:
                df        = self._load_csv_safely(filepath)
                is_anti   = "anti" in clean_task.lower()
                axis_char = 'h' if axis_type == "Horizontal" else 'v'

                time_col   = next((c for c in df.columns if 'time' in c or c == 't'), df.columns[0])
                time_val   = df[time_col].dropna().values
                current_fs = 1.0 / np.mean(np.diff(time_val)) if len(time_val) > 1 else 120.0

                target_col = next(
                    (c for c in df.columns if f'target{axis_char}' in c or f'target_{axis_char}' in c), None
                )
                if not target_col: continue
                if is_anti: df[target_col] = df[target_col] * -1

                left_col  = next((c for c in df.columns if c == f'l{axis_char}'), None)
                right_col = next((c for c in df.columns if c == f'r{axis_char}'), None)
                if not left_col or not right_col:
                    continue

                cwt_epochs = self._extract_binocular_epochs(
                    df, target_col, left_col, right_col, current_fs
                )
                if cwt_epochs:
                    self.data_store[group][subject_id][clean_task].extend(cwt_epochs)
                    processed += 1
            except Exception as e:
                print(f"[!] Skipped {filepath.name}: {e}")

        print(f"[*] Processed {processed} CSV files")

---

## Layer 2 — PyTorch Dataset Bridge

### 2.1 Normalization

Each CWT tensor $\mathbf{Z}_k \in \mathbb{R}^{2 \times F \times T}$ is normalized independently per channel:

$$\hat{Z}_k^{(c)} = \frac{Z_k^{(c)} - \mu_k^{(c)}}{\sigma_k^{(c)} + \epsilon}, \quad c \in \{\mathrm{Re},\, \mathrm{Im}\}$$

where $\mu_k^{(c)}$ and $\sigma_k^{(c)}$ are the mean and standard deviation over all $(f, t)$ positions of channel $c$ in epoch $k$. Per-channel normalization is preferred over joint normalization to prevent the larger-magnitude real channel from dominating the imaginary channel, which carries complementary phase information.

The additive $\epsilon = 10^{-8}$ prevents division by zero for flat epochs (e.g., missing eye data).

### 2.2 Subject-ID Tracking for Leakage-Free Evaluation

Each sample in the dataset carries a `subject_id` identifier (the recording timestamp folder, unique per patient). This is critical: **epoch-level random splitting would allow multiple epochs from the same subject to appear in both the training and validation sets**, causing the model to learn subject-specific idiosyncrasies rather than generalizable MCI biomarkers — a form of data leakage that inflates validation accuracy.

The `subject_ids` list enables the `ModelTrainer` to perform a proper **subject-level stratified split**, where all epochs from a given subject appear exclusively in either the training set or the validation set — never both.

In [ ]:
# =========================================================================
# [Layer 2] PyTorch Dataset Bridge
# Enhancements:
#   - Updated for 3-level data_store (group→subject→task→tensors)
#   - 4-channel normalization [Re_L, Im_L, Re_R, Im_R]
# =========================================================================
class VOG_CWT_Dataset(Dataset):
    def __init__(self, data_store):
        self.X           = []
        self.y           = []
        self.subject_ids = []

        for group, subjects in data_store.items():
            label = 0 if group == "HC_csv_24_25" else 1
            for subject_id, tasks in subjects.items():
                for task, tensors in tasks.items():       # 3-level: no eye dimension
                    for tensor in tensors:                # [4, freq, time]
                        norm_chs = []
                        for ch in range(tensor.shape[0]):
                            d = tensor[ch]
                            norm_chs.append((d - np.mean(d)) / (np.std(d) + 1e-8))
                        self.X.append(np.stack(norm_chs, axis=0))
                        self.y.append(label)
                        self.subject_ids.append(subject_id)

        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(self.y, dtype=torch.long)

    def __len__(self):  return len(self.y)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]


# =========================================================================
# [Layer 2b] SpecAugment Wrapper
# Applied only to training subsets; validation/inference use raw tensors.
# Frequency masking + time masking applied on-the-fly per batch.
# =========================================================================
class AugmentedSubset(Dataset):
    """
    Wraps any Subset and applies SpecAugment-style masking on-the-fly.
    f_mask_ratio: fraction of frequency bins to zero out per sample
    t_mask_ratio: fraction of time bins to zero out per sample
    """
    def __init__(self, subset, f_mask_ratio=0.2, t_mask_ratio=0.2):
        self.subset = subset
        x0, _ = subset[0]
        _, freq_bins, time_bins = x0.shape
        self.f_mask_size = max(1, int(freq_bins * f_mask_ratio))
        self.t_mask_size = max(1, int(time_bins * t_mask_ratio))

    def __len__(self): return len(self.subset)

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        x = x.clone()
        freq_bins, time_bins = x.shape[1], x.shape[2]
        # Frequency masking: zero a random contiguous band
        f0 = random.randint(0, freq_bins - self.f_mask_size)
        x[:, f0:f0 + self.f_mask_size, :] = 0.0
        # Time masking: zero a random contiguous window
        t0 = random.randint(0, time_bins - self.t_mask_size)
        x[:, :, t0:t0 + self.t_mask_size] = 0.0
        return x, y

---

## Layer 3 — EdgeCWTClassifier (CNN–CBAM Hybrid)

### Architecture Summary

The model is a lightweight CNN designed for real-time inference on the Jetson AGX Orin (Ampere GPU, 16–32 GB unified memory). It treats the 2-channel CWT scalogram as a 2D image with two input channels — analogous to how an RGB image has 3 channels — and applies a hierarchical feature extraction with attention gating.

| Stage | Operation | Output shape | Parameters |
|-------|-----------|-------------|------------|
| Input | — | `[B, 2, 40, 100]` | — |
| Block 1 | Conv(2→16, 3×3) + BN + ReLU + MaxPool(2×2) | `[B, 16, 20, 50]` | 304 |
| CBAM 1 | Channel(16) + Spatial(7×7) | `[B, 16, 20, 50]` | 386 |
| Block 2 | DW-Conv(16, 3×3) + PW-Conv(16→32, 1×1) + BN + ReLU + MaxPool | `[B, 32, 10, 25]` | 688 |
| CBAM 2 | Channel(32) + Spatial(7×7) | `[B, 32, 10, 25]` | 1,346 |
| Block 3 | DW-Conv(32, 3×3) + PW-Conv(32→64, 1×1) + BN + ReLU | `[B, 64, 10, 25]` | 2,368 |
| CBAM 3 | Channel(64) + Spatial(7×7) | `[B, 64, 10, 25]` | 4,994 |
| GAP | AdaptiveAvgPool(1×1) | `[B, 64]` | — |
| Head | Dropout(0.3) + FC(64→32) + ReLU + FC(32→2) | `[B, 2]` | 2,146 |

**Total trainable parameters: ~12,000** — intentionally minimal for a dataset of ~74 subjects.

### Design Rationale

**Depthwise-Separable Convolutions (Blocks 2 & 3):** Factorize a standard $k \times k$ convolution into a depthwise convolution (one filter per input channel) followed by a $1 \times 1$ pointwise convolution (channel mixing). For a $C_{in} \to C_{out}$ layer with kernel $k$, the parameter count reduces from $k^2 C_{in} C_{out}$ to $k^2 C_{in} + C_{in} C_{out}$, a reduction factor of $\approx 8\times$ at $k=3$.

**CBAM after Block 3, before GAP:** The Global Average Pooling collapses spatial dimensions to $1 \times 1$, destroying spatial information. Placing CBAM *before* GAP ensures the spatial attention operates on a feature map still carrying $(f, t)$ structure ($10 \times 25$ pixels), making the attention weights interpretable as a time-frequency saliency mask.

**Dropout(0.3):** Applied before the classifier head to regularize the final representation, critical given the small number of subjects.

In [ ]:
# =========================================================================
# [Layer 3] Edge AI Model: CNN-CBAM Hybrid (enhanced)
#
# Enhancements:
#   - in_channels=4 (binocular: Re_L, Im_L, Re_R, Im_R)
#   - Residual connections on Blocks 2 & 3 (skip connection via 1×1 conv)
#
# Input:  [B, 4, 40, 100]
# Block 1: Conv(4→16)  + BN + ReLU + MaxPool       → [B, 16, 20, 50]  + CBAM
# Block 2: DW+PW(16→32) + residual + MaxPool        → [B, 32, 10, 25]  + CBAM
# Block 3: DW+PW(32→64) + residual                  → [B, 64, 10, 25]  + CBAM → GAP
# Head:   Dropout(0.3) + FC(64→32) + ReLU + FC(32→2)
# =========================================================================
class EdgeCWTClassifier(nn.Module):
    def __init__(self, num_classes=2, in_channels=4):
        super().__init__()

        # Block 1: standard conv — no residual (input channels vary)
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
        )
        self.cbam1 = CBAM(16)

        # Block 2: DW-Sep + residual skip
        self.block2_main = nn.Sequential(
            nn.Conv2d(16, 16, kernel_size=3, padding=1, groups=16),  # depthwise
            nn.Conv2d(16, 32, kernel_size=1),                         # pointwise
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2),
        )
        self.block2_skip = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=1, bias=False),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2),
        )
        self.cbam2 = CBAM(32)

        # Block 3: DW-Sep + residual skip (no pool — preserve spatial for CBAM)
        self.block3_main = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, padding=1, groups=32),  # depthwise
            nn.Conv2d(32, 64, kernel_size=1),                         # pointwise
            nn.BatchNorm2d(64),
        )
        self.block3_skip = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=1, bias=False),
            nn.BatchNorm2d(64),
        )
        self.cbam3 = CBAM(64)
        self.gap   = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        # Block 1 (no residual)
        x = self.cbam1(self.block1(x))
        # Block 2 with residual
        x = self.cbam2(F.relu(self.block2_main(x) + self.block2_skip(x), inplace=True))
        # Block 3 with residual
        x = self.cbam3(F.relu(self.block3_main(x) + self.block3_skip(x), inplace=True))
        # GAP → flatten → classify
        x = self.gap(x)
        return self.classifier(torch.flatten(x, 1))

---

## Layer 4a — Model Trainer

### 4.1 Subject-Level Stratified Split

With $N \approx 74$ total subjects, preserving **statistical independence** between the training and validation cohorts is paramount. The split is performed at the **subject level**, not the epoch level:

1. Group all epoch indices by subject ID: $\mathcal{I}_s = \{i : \text{subject\_id}[i] = s\}$
2. Separately shuffle the HC subjects $\mathcal{S}_{\mathrm{HC}}$ and MCI subjects $\mathcal{S}_{\mathrm{MCI}}$
3. Assign the first $\lfloor 0.2 |\mathcal{S}_c| \rfloor$ subjects per class to the validation set
4. Map subject IDs back to epoch indices via $\mathcal{I}_s$

This guarantees that no information about a validation subject leaks into the training set through shared epochs — equivalent to the statistical requirement of independent and identically distributed (i.i.d.) test samples.

### 4.2 Weighted Cross-Entropy Loss

The dataset exhibits a class imbalance of approximately **1:1.6 (HC:MCI)**. Standard cross-entropy optimizes the average log-loss uniformly over samples, which on an imbalanced dataset biases the model toward the majority class (MCI). The weighted cross-entropy applies **inverse-frequency weighting**:

$$w_c = \frac{N}{K \cdot N_c}, \quad \mathcal{L}_{\mathrm{WCE}} = -\frac{1}{N}\sum_{i=1}^{N} w_{y_i} \log p(y_i \mid \mathbf{x}_i)$$

where $N$ is the total number of training epochs, $K=2$ is the number of classes, and $N_c$ is the count of class $c$. This ensures that a misclassification of the minority class (HC) incurs proportionally higher loss, encouraging the model to maintain sensitivity for both classes.

### 4.3 Optimization and Learning Rate Schedule

The optimizer is **AdamW** (Adam with decoupled weight decay), which applies L2 regularization directly to the weights rather than to the gradient update — correcting a flaw in the original Adam implementation and providing more reliable regularization:

$$\theta_{t+1} = \theta_t - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} - \alpha \lambda \theta_t$$

The learning rate follows a **Cosine Annealing** schedule:

$$\alpha_t = \alpha_{\min} + \frac{1}{2}(\alpha_0 - \alpha_{\min})\left(1 + \cos\frac{\pi t}{T_{\max}}\right)$$

with $\alpha_0 = 10^{-3}$ and $T_{\max} = $ `epochs`. This avoids the sharp learning rate drops of step-decay schedules and has been shown to improve generalization by exploring flat minima in the loss landscape.

### 4.4 Mixed-Precision Training (FP16)

On the Jetson AGX Orin's Ampere Tensor Cores, FP16 arithmetic runs at up to $2\times$ the throughput of FP32. PyTorch's `torch.amp.autocast` automatically casts eligible operations (convolutions, matrix multiplications) to FP16 while maintaining FP32 for numerically sensitive operations (batch norm, loss computation). The `GradScaler` multiplies the loss by a large factor before backpropagation to prevent FP16 gradient underflow, then unscales before the optimizer step.

In [ ]:
# =========================================================================
# [Layer 4a] Model Trainer
# Enhancements:
#   - AugmentedSubset applied to training split only
#   - Label smoothing (0.1) in CrossEntropyLoss
#   - Early stopping with patience=10
#   - Default epochs raised to 50
#   - Device priority: CUDA → MPS (Apple Silicon) → CPU
# =========================================================================
class ModelTrainer:
    def __init__(self, model, device="auto"):
        if device in ("auto", "cuda"):
            if torch.cuda.is_available():            self.device = torch.device("cuda")
            elif torch.backends.mps.is_available():  self.device = torch.device("mps")
            else:                                    self.device = torch.device("cpu")
        else:
            self.device = torch.device(device)
        self.model     = model.to(self.device)
        self.use_amp   = self.device.type == "cuda"
        self.scaler    = torch.amp.GradScaler('cuda') if self.use_amp else None
        self.optimizer = optim.AdamW(self.model.parameters(), lr=1e-3, weight_decay=1e-4)

    def _subject_stratified_split(self, dataset, val_ratio=0.2, seed=42):
        random.seed(seed)
        subject_to_idx = defaultdict(list)
        for i, sid in enumerate(dataset.subject_ids):
            subject_to_idx[sid].append(i)

        hc_subjects  = [s for s in subject_to_idx if dataset.y[subject_to_idx[s][0]].item() == 0]
        mci_subjects = [s for s in subject_to_idx if dataset.y[subject_to_idx[s][0]].item() == 1]
        random.shuffle(hc_subjects); random.shuffle(mci_subjects)

        hc_val_n  = max(1, int(len(hc_subjects)  * val_ratio))
        mci_val_n = max(1, int(len(mci_subjects) * val_ratio))

        val_subjects   = set(hc_subjects[:hc_val_n]   + mci_subjects[:mci_val_n])
        train_subjects = set(hc_subjects[hc_val_n:]   + mci_subjects[mci_val_n:])

        train_idx = [i for i, s in enumerate(dataset.subject_ids) if s in train_subjects]
        val_idx   = [i for i, s in enumerate(dataset.subject_ids) if s in val_subjects]

        n_hc  = sum(1 for s in train_subjects if dataset.y[subject_to_idx[s][0]].item() == 0)
        n_mci = sum(1 for s in train_subjects if dataset.y[subject_to_idx[s][0]].item() == 1)
        print(f"[*] Train subjects: HC_csv_24_25={n_hc}, MCI={n_mci} | "
              f"Val subjects: HC_csv_24_25={hc_val_n}, MCI={mci_val_n}")

        return Subset(dataset, train_idx), Subset(dataset, val_idx)

    def train_model(self, dataset, epochs=50, batch_size=32, patience=10):
        print(f"[*] 학습 시작 (디바이스: {self.device})")

        train_raw, val_subset = self._subject_stratified_split(dataset)
        # SpecAugment applied to training data only
        train_subset = AugmentedSubset(train_raw, f_mask_ratio=0.2, t_mask_ratio=0.2)
        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True,  drop_last=True)
        val_loader   = DataLoader(val_subset,   batch_size=batch_size, shuffle=False)

        label_counts  = Counter(dataset.y.tolist())
        total         = len(dataset)
        class_weights = torch.tensor(
            [total / (2 * label_counts[i]) for i in range(2)], dtype=torch.float32
        ).to(self.device)
        # Label smoothing: prevents overconfidence, improves calibration
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        print(f"[*] Class weights — HC_csv_24_25: {class_weights[0]:.3f}, MCI: {class_weights[1]:.3f}")

        scheduler      = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=epochs)
        best_val_loss  = float('inf')
        no_improve     = 0

        for epoch in range(epochs):
            # --- Training ---
            self.model.train()
            train_loss = 0.0; train_correct = 0; train_total = 0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                self.optimizer.zero_grad()
                with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                    outputs = self.model(inputs)
                    loss    = criterion(outputs, labels)
                if self.use_amp:
                    self.scaler.scale(loss).backward()
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    loss.backward(); self.optimizer.step()
                train_loss    += loss.item() * inputs.size(0)
                _, predicted   = outputs.max(1)
                train_total   += labels.size(0)
                train_correct += predicted.eq(labels).sum().item()

            # --- Validation ---
            self.model.eval()
            val_loss = 0.0; val_correct = 0; val_total = 0
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(self.device), labels.to(self.device)
                    with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                        outputs = self.model(inputs)
                        loss    = criterion(outputs, labels)
                    val_loss    += loss.item() * inputs.size(0)
                    _, predicted = outputs.max(1)
                    val_total   += labels.size(0)
                    val_correct += predicted.eq(labels).sum().item()

            train_acc    = 100. * train_correct / train_total
            val_acc      = 100. * val_correct   / val_total
            val_loss_avg = val_loss / val_total
            scheduler.step()

            print(f"Epoch [{epoch+1:02d}/{epochs}] "
                  f"| Train Acc: {train_acc:.2f}% "
                  f"| Val Acc: {val_acc:.2f}% "
                  f"| Val Loss: {val_loss_avg:.4f} "
                  f"| LR: {scheduler.get_last_lr()[0]:.2e}")

            if val_loss_avg < best_val_loss:
                best_val_loss = val_loss_avg
                no_improve    = 0
                torch.save(self.model.state_dict(), 'best_edge_cwt_model.pth')
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"[Early stop] 개선 없음 {patience} epochs — 학습 종료.")
                    break

        print(f"[+] 학습 완료. Best val loss: {best_val_loss:.4f} → 'best_edge_cwt_model.pth' 저장됨.")

---

## Layer 4b — Jetson Inference Engine

### Epoch-Level Soft-Voting Ensemble

A single CSV file from a patient contains multiple saccade trials. Rather than classifying on a single epoch (which is noisy), the engine extracts $K$ CWT epochs from the file, runs inference on each, and aggregates predictions via **soft-voting** (averaging posterior probabilities):

$$\hat{p}(y = 1 \mid \mathcal{X}) = \frac{1}{K} \sum_{k=1}^{K} p(y = 1 \mid \mathbf{z}_k; \hat{\theta})$$

where $\mathbf{z}_k$ is the normalized CWT tensor for epoch $k$ and $\hat{\theta}$ are the trained model parameters. Under the assumption that individual epochs are conditionally independent given the patient's clinical state, this estimator has variance $\mathrm{Var}[\hat{p}] = \sigma^2_k / K$, improving robustness by a factor of $\sqrt{K}$ compared to single-epoch classification.

The final diagnosis threshold is $\hat{y} = \mathbf{1}[\hat{p}(y=1) > 0.5]$, with the MCI confidence score reported as a percentage.

In [ ]:
# =========================================================================
# [Layer 4b] Jetson Inference Engine (binocular-aware)
# =========================================================================
class JetsonInferenceEngine:
    def __init__(self, model_path, pipeline_config, device="auto"):
        if device in ("auto", "cuda"):
            if torch.cuda.is_available():            self.device = torch.device("cuda")
            elif torch.backends.mps.is_available():  self.device = torch.device("mps")
            else:                                    self.device = torch.device("cpu")
        else:
            self.device = torch.device(device)
        self.use_amp  = self.device.type == "cuda"
        self.pipeline = EventLockedCWTPipeline(**pipeline_config)
        self.model    = EdgeCWTClassifier(num_classes=2, in_channels=4)
        if os.path.exists(model_path):
            self.model.load_state_dict(
                torch.load(model_path, map_location=self.device, weights_only=True)
            )
        self.model.to(self.device).eval()
        self.classes = ["Healthy Control (HC_csv_24_25)", "Mild Cognitive Impairment (MCI)"]

    def infer_csv(self, csv_filepath: Path):
        print(f"\n[*] 추론 시작: {csv_filepath.name}")
        df = self.pipeline._load_csv_safely(csv_filepath)

        axis_char  = 'h' if 'Horizontal' in csv_filepath.name else 'v'
        target_col = next(
            (c for c in df.columns if f'target{axis_char}' in c or f'target_{axis_char}' in c), None
        )
        left_col  = next((c for c in df.columns if c == f'l{axis_char}'), None)
        right_col = next((c for c in df.columns if c == f'r{axis_char}'), None)

        if not target_col or not left_col or not right_col:
            return "추론 불가: Target 또는 Eye 컬럼을 찾을 수 없습니다."

        time_col   = next((c for c in df.columns if 'time' in c or c == 't'), df.columns[0])
        time_val   = df[time_col].dropna().values
        current_fs = 1.0 / np.mean(np.diff(time_val)) if len(time_val) > 1 else 120.0

        cwt_epochs = self.pipeline._extract_binocular_epochs(
            df, target_col, left_col, right_col, current_fs
        )
        if not cwt_epochs:
            return "추론 불가: 유효한 Saccade 이벤트가 없습니다."

        # Normalize each channel independently
        input_tensors = []
        for tensor in cwt_epochs:   # [4, freq, time]
            norm_chs = []
            for ch in range(tensor.shape[0]):
                d = tensor[ch]
                norm_chs.append((d - np.mean(d)) / (np.std(d) + 1e-8))
            input_tensors.append(np.stack(norm_chs, axis=0))

        inputs = torch.tensor(np.array(input_tensors), dtype=torch.float32).to(self.device)

        with torch.no_grad():
            with torch.amp.autocast(device_type=self.device.type, enabled=self.use_amp):
                outputs       = self.model(inputs)
                probabilities = F.softmax(outputs, dim=1)

        mean_prob       = probabilities.mean(dim=0)
        predicted_class = torch.argmax(mean_prob).item()
        mci_confidence  = mean_prob[1].item() * 100

        print(f"[>] Saccade 이벤트 수: {len(cwt_epochs)}")
        print(f"[>] 앙상블 진단: {self.classes[predicted_class]}")
        print(f"[>] MCI 확률: {mci_confidence:.2f}%")
        return predicted_class, mci_confidence

---

## Layer 5 — XAI Visualizer: SPM-style Difference Maps

### Statistical Parametric Mapping in the Time-Frequency Domain

Statistical Parametric Mapping (SPM), originally developed for fMRI and EEG neuroimaging analysis, identifies voxels (or, here, time-frequency bins) where the observed signal differs significantly between experimental groups. This layer implements the **group contrast map** component of SPM:

**Step 1 — Group-mean scalogram in decibels.**
For each group $g \in \{\mathrm{HC}, \mathrm{MCI}\}$, compute the mean CWT power across all $N_g$ epochs:

$$\bar{P}_g(f, t) = \frac{1}{N_g} \sum_{i=1}^{N_g} \left|W_\psi[\tilde{e}_i](f, t)\right|^2$$

Convert to decibels to compress the dynamic range and work on a perceptually linear scale:

$$\overline{\mathrm{CWT}}_g(f, t)_{\mathrm{dB}} = 10 \log_{10}\bigl(\bar{P}_g(f, t) + \epsilon\bigr)$$

**Step 2 — Group contrast (Difference Map).**
The difference map is the MCI-minus-HC contrast:

$$\Delta(f, t) = \overline{\mathrm{CWT}}_{\mathrm{MCI}}(f, t)_{\mathrm{dB}} - \overline{\mathrm{CWT}}_{\mathrm{HC}}(f, t)_{\mathrm{dB}}$$

**Interpretation of $\Delta(f, t)$:**
- **Red regions** ($\Delta > 0$): MCI patients exhibit *higher* gaze error power at frequency $f$ and time $t$ post-stimulus. This indicates oscillatory instability or prolonged tracking failure specific to MCI.
- **Blue regions** ($\Delta < 0$): HC subjects exhibit higher power, indicating more vigorous corrective saccades or faster re-fixation.

The diverging colormap (`RdBu_r`) is centered at zero, with the white dashed vertical line marking $t = 0$ (stimulus onset). The diagram provides **clinical face validity** for the model before any ML training, demonstrating that the raw signal features are group-discriminant in physiologically plausible time-frequency regions.

> **Note on multiple comparisons:** The current implementation plots the raw mean difference without thresholding. For formal hypothesis testing (e.g., cluster-based permutation tests or FDR correction across time-frequency bins), a statistical thresholding step would be applied on top of $\Delta(f, t)$ — a natural extension for a full neuroimaging-style SPM analysis.

In [ ]:
# =========================================================================
# [Layer 5] XAI Visualizer: SPM-style Difference Maps
# Updated for 3-level data_store + 4-channel binocular tensors
# Power = mean of left-eye power + right-eye power
# =========================================================================
class XAIVisualizer:
    def __init__(self, pipeline: EventLockedCWTPipeline):
        self.frequencies = pipeline.frequencies
        self.time_bins   = pipeline.target_time_bins
        self.pre_sec     = pipeline.pre_sec
        self.post_sec    = pipeline.post_sec

    def _compute_group_mean_db(self, data_store):
        group_powers = defaultdict(list)
        for group, subjects in data_store.items():
            for subject_id, tasks in subjects.items():
                for task, tensors in tasks.items():   # 3-level: no eye dimension
                    for tensor in tensors:            # [4, freq, time]
                        # Average binocular power: (|L|² + |R|²) / 2
                        power = (tensor[0]**2 + tensor[1]**2 +
                                 tensor[2]**2 + tensor[3]**2) / 2.0
                        group_powers[group].append(power)
        return {
            group: 10 * np.log10(np.mean(np.array(powers), axis=0) + 1e-10)
            for group, powers in group_powers.items()
        }

    def plot_difference_map(self, data_store, save_path=None):
        group_mean_db = self._compute_group_mean_db(data_store)
        if 'HC_csv_24_25' not in group_mean_db or 'MCI' not in group_mean_db:
            print("[!] HC와 MCI 데이터가 모두 필요합니다.")
            return

        diff_map  = group_mean_db['MCI'] - group_mean_db['HC_csv_24_25']
        time_axis = np.linspace(-self.pre_sec, self.post_sec, self.time_bins)
        extent    = [time_axis[0], time_axis[-1], self.frequencies[0], self.frequencies[-1]]

        fig, axes = plt.subplots(1, 3, figsize=(18, 5))
        configs = [
            (group_mean_db['HC_csv_24_25'],  'HC_csv_24_25 Mean CWT (dB)',          'viridis', None,   None),
            (group_mean_db['MCI'], 'MCI Mean CWT (dB)',         'viridis', None,   None),
            (diff_map,             'Difference: MCI \u2212 HC_csv_24_25 (dB)', 'RdBu_r',
             -np.abs(diff_map).max(), np.abs(diff_map).max()),
        ]
        for ax, (data, title, cmap, vmin, vmax) in zip(axes, configs):
            im = ax.imshow(data, aspect='auto', origin='lower', extent=extent,
                           cmap=cmap, vmin=vmin, vmax=vmax)
            ax.axvline(x=0, color='white', linestyle='--', linewidth=1.2, alpha=0.8)
            ax.set_title(title, fontsize=12)
            ax.set_xlabel('Time (s)'); ax.set_ylabel('Frequency (Hz)')
            cbar = plt.colorbar(im, ax=ax)
            cbar.set_label('dB' if 'Difference' not in title else '\u0394dB')

        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"[+] Difference Map 저장됨: {save_path}")
        plt.show()

---

## Layer 6 — Leave-One-Subject-Out Cross-Validation (LOSO-CV)

### Motivation

The 80/20 subject-level split used in `ModelTrainer` leaves only ~6 subjects for validation, producing high-variance accuracy estimates (a single misclassification shifts accuracy by ~16%). With $N = 37$ subjects, the statistically appropriate evaluation is **Leave-One-Subject-Out Cross-Validation (LOSO-CV)**:

1. For each subject $s \in \{1, \ldots, N\}$, hold out **all epochs** belonging to $s$
2. Train a fresh model on the remaining $N-1$ subjects' epochs
3. Aggregate per-epoch posteriors via soft-voting to produce a **subject-level prediction**:

$$\hat{p}_s = \frac{1}{K_s} \sum_{k=1}^{K_s} p(y=1 \mid \mathbf{z}_k;\,\hat{\theta}^{(-s)})$$

4. Record $(\hat{p}_s, y_s)$ — the subject-level MCI probability and true label

This yields exactly $N$ independent subject-level predictions, each from a model that has **never seen** that subject. The resulting metrics are unbiased estimates of generalisation performance.

### Clinical Metrics

Binary accuracy is insufficient for a clinical screening tool. The evaluator reports:

| Metric | Formula | Clinical meaning |
|--------|---------|-----------------|
| **Sensitivity** | $\mathrm{TP}/(\mathrm{TP}+\mathrm{FN})$ | Fraction of MCI patients correctly flagged |
| **Specificity** | $\mathrm{TN}/(\mathrm{TN}+\mathrm{FP})$ | Fraction of HC correctly cleared |
| **Balanced Accuracy** | $\frac{1}{2}(\text{Sens}+\text{Spec})$ | Accuracy adjusted for class imbalance |
| **AUROC** | $\int \mathrm{TPR}\,d(\mathrm{FPR})$ | Threshold-independent discrimination |

In [ ]:
# =========================================================================
# [Layer 6] LOSO-CV Evaluator (enhanced)
# Enhancements:
#   - AugmentedSubset applied to each training fold
#   - Label smoothing (0.1) in CrossEntropyLoss
#   - Default epochs raised to 50
#   - Threshold optimisation: finds threshold maximising balanced accuracy
# =========================================================================
class LOSOCrossValidator:
    def __init__(self, dataset, device="auto", epochs=50, batch_size=32):
        if device in ("auto", "cuda"):
            if torch.cuda.is_available():            self.device = torch.device("cuda")
            elif torch.backends.mps.is_available():  self.device = torch.device("mps")
            else:                                    self.device = torch.device("cpu")
        else:
            self.device = torch.device(device)
        self.dataset    = dataset
        self.epochs     = epochs
        self.batch_size = batch_size

    # ------------------------------------------------------------------
    @staticmethod
    def _auroc(true_labels, scores):
        order    = np.argsort(scores)[::-1]
        y_sorted = true_labels[order]
        n_pos    = np.sum(true_labels == 1)
        n_neg    = np.sum(true_labels == 0)
        if n_pos == 0 or n_neg == 0:
            return float('nan')
        tp = fp = 0
        tprs, fprs = [0.0], [0.0]
        for label in y_sorted:
            if label == 1: tp += 1
            else:          fp += 1
            tprs.append(tp / n_pos)
            fprs.append(fp / n_neg)
        return float(np.trapezoid(tprs, fprs))

    # ------------------------------------------------------------------
    def _train_one_fold(self, train_idx):
        train_raw    = Subset(self.dataset, train_idx)
        # SpecAugment on training fold only
        train_subset = AugmentedSubset(train_raw, f_mask_ratio=0.2, t_mask_ratio=0.2)
        train_loader = DataLoader(train_subset, batch_size=self.batch_size,
                                  shuffle=True, drop_last=True)

        model     = EdgeCWTClassifier(num_classes=2, in_channels=4).to(self.device)
        use_amp   = self.device.type == "cuda"
        scaler    = torch.amp.GradScaler('cuda') if use_amp else None
        optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.epochs)

        label_counts  = Counter(self.dataset.y[train_idx].tolist())
        total         = len(train_idx)
        class_weights = torch.tensor(
            [total / (2 * label_counts[i]) for i in range(2)], dtype=torch.float32
        ).to(self.device)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

        for _ in range(self.epochs):
            model.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(self.device), labels.to(self.device)
                optimizer.zero_grad()
                with torch.amp.autocast(device_type=self.device.type, enabled=use_amp):
                    loss = criterion(model(inputs), labels)
                if use_amp:
                    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                else:
                    loss.backward(); optimizer.step()
            scheduler.step()
        return model

    # ------------------------------------------------------------------
    def _infer_subject(self, model, test_idx):
        test_loader = DataLoader(Subset(self.dataset, test_idx),
                                 batch_size=self.batch_size, shuffle=False)
        all_probs = []
        model.eval()
        use_amp = self.device.type == "cuda"
        with torch.no_grad():
            for inputs, _ in test_loader:
                inputs = inputs.to(self.device)
                with torch.amp.autocast(device_type=self.device.type, enabled=use_amp):
                    probs = F.softmax(model(inputs), dim=1)
                all_probs.append(probs.cpu())
        mean_prob = torch.cat(all_probs, dim=0).mean(dim=0)
        return torch.argmax(mean_prob).item(), mean_prob[1].item()

    # ------------------------------------------------------------------
    def run(self):
        subject_to_idx = defaultdict(list)
        for i, sid in enumerate(self.dataset.subject_ids):
            subject_to_idx[sid].append(i)

        all_subjects = list(subject_to_idx.keys())
        N            = len(all_subjects)
        all_indices  = set(range(len(self.dataset)))

        records = []
        print(f"[*] LOSO-CV 시작 — {N} subjects, device={self.device}\n")

        for fold, subject in enumerate(all_subjects):
            test_idx   = subject_to_idx[subject]
            train_idx  = list(all_indices - set(test_idx))
            true_label = self.dataset.y[test_idx[0]].item()

            model = self._train_one_fold(train_idx)
            pred_label, mci_prob = self._infer_subject(model, test_idx)

            mark     = "✓" if true_label == pred_label else "✗"
            true_str = "MCI" if true_label else "HC_csv_24_25 "
            pred_str = "MCI" if pred_label else "HC_csv_24_25 "
            print(f"  [{fold+1:02d}/{N}] {mark}  True={true_str}  Pred={pred_str}  "
                  f"MCI_prob={mci_prob:.3f}  epochs={len(test_idx):4d}  ({subject[:24]})")
            records.append((subject, true_label, pred_label, mci_prob, len(test_idx)))

        return self._report(records)

    # ------------------------------------------------------------------
    def _report(self, records):
        _, true_arr, pred_arr, prob_arr, _ = zip(*records)
        true_arr = np.array(true_arr)
        pred_arr = np.array(pred_arr)
        prob_arr = np.array(prob_arr)

        def _metrics(ta, pa):
            tp = int(np.sum((ta==1)&(pa==1))); tn = int(np.sum((ta==0)&(pa==0)))
            fp = int(np.sum((ta==0)&(pa==1))); fn = int(np.sum((ta==1)&(pa==0)))
            acc  = (tp+tn)/len(ta)
            sens = tp/(tp+fn) if tp+fn>0 else 0.0
            spec = tn/(tn+fp) if tn+fp>0 else 0.0
            return tp, tn, fp, fn, acc, sens, spec

        tp, tn, fp, fn, accuracy, sensitivity, specificity = _metrics(true_arr, pred_arr)
        balanced_acc = (sensitivity + specificity) / 2
        auroc        = self._auroc(true_arr, prob_arr)

        # ── Threshold optimisation ────────────────────────────────────
        best_t, best_bacc = 0.5, balanced_acc
        for t in np.linspace(0.1, 0.9, 81):
            pa_t = (prob_arr > t).astype(int)
            _, _, _, _, _, s_, sp_ = _metrics(true_arr, pa_t)
            b_ = (s_ + sp_) / 2
            if b_ > best_bacc:
                best_bacc = b_; best_t = t
        pa_opt = (prob_arr > best_t).astype(int)
        tp_o, tn_o, fp_o, fn_o, acc_o, sens_o, spec_o = _metrics(true_arr, pa_opt)

        print("\n" + "=" * 56)
        print("  LOSO-CV  Summary")
        print("=" * 56)
        n_hc  = int(np.sum(true_arr == 0))
        n_mci = int(np.sum(true_arr == 1))
        print(f"  Subjects     : {len(records)}  (HC_csv_24_25={n_hc}, MCI={n_mci})")
        print(f"  ── Threshold = 0.50 (default) ──")
        print(f"  Accuracy     : {accuracy:.3f}   ({tp+tn}/{len(records)})")
        print(f"  Sensitivity  : {sensitivity:.3f}   TP={tp}  FN={fn}")
        print(f"  Specificity  : {specificity:.3f}   TN={tn}  FP={fp}")
        print(f"  Balanced Acc : {balanced_acc:.3f}")
        print(f"  AUROC        : {auroc:.3f}")
        print(f"  ── Threshold = {best_t:.2f} (optimised for balanced acc) ──")
        print(f"  Accuracy     : {acc_o:.3f}   ({tp_o+tn_o}/{len(records)})")
        print(f"  Sensitivity  : {sens_o:.3f}   TP={tp_o}  FN={fn_o}")
        print(f"  Specificity  : {spec_o:.3f}   TN={tn_o}  FP={fp_o}")
        print(f"  Balanced Acc : {best_bacc:.3f}")
        print("=" * 56)

        # ── ROC curve + probability strip ────────────────────────────
        order    = np.argsort(prob_arr)[::-1]
        y_sorted = true_arr[order]
        n_pos    = int(np.sum(true_arr == 1))
        n_neg    = int(np.sum(true_arr == 0))
        tp_r = fp_r = 0
        tprs, fprs = [0.0], [0.0]
        for lbl in y_sorted:
            if lbl == 1: tp_r += 1
            else:        fp_r += 1
            tprs.append(tp_r / n_pos); fprs.append(fp_r / n_neg)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        axes[0].plot(fprs, tprs, 'b-o', markersize=5, label=f'AUROC = {auroc:.3f}')
        axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
        axes[0].fill_between(fprs, tprs, alpha=0.1)
        axes[0].set_xlabel('1 − Specificity (FPR)', fontsize=12)
        axes[0].set_ylabel('Sensitivity (TPR)', fontsize=12)
        axes[0].set_title('LOSO-CV ROC Curve', fontsize=13)
        axes[0].legend(fontsize=11); axes[0].grid(alpha=0.3)

        hc_probs  = prob_arr[true_arr == 0]
        mci_probs = prob_arr[true_arr == 1]
        axes[1].scatter(hc_probs,  np.zeros_like(hc_probs)  + 0.15,
                        color='steelblue', s=90, label=f'HC_csv_24_25  (n={n_neg})', zorder=3, alpha=0.85)
        axes[1].scatter(mci_probs, np.zeros_like(mci_probs) + 0.85,
                        color='tomato',    s=90, label=f'MCI (n={n_pos})', zorder=3, alpha=0.85)
        axes[1].axvline(0.5,    color='gray',   linestyle='--', linewidth=1.2, label='default=0.5')
        axes[1].axvline(best_t, color='orange', linestyle=':',  linewidth=1.5,
                        label=f'optimal={best_t:.2f}')
        axes[1].set_xlim(-0.05, 1.05); axes[1].set_ylim(0, 1)
        axes[1].set_yticks([0.15, 0.85]); axes[1].set_yticklabels(['HC_csv_24_25', 'MCI'], fontsize=12)
        axes[1].set_xlabel('Soft-vote MCI probability', fontsize=12)
        axes[1].set_title('Per-subject MCI probability', fontsize=13)
        axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

        plt.tight_layout()
        plt.savefig('../loso_results.png', dpi=150, bbox_inches='tight')
        print("[+] Saved: ../loso_results.png")
        plt.show()

        return dict(accuracy=accuracy, sensitivity=sensitivity, specificity=specificity,
                    balanced_accuracy=balanced_acc, auroc=auroc,
                    opt_threshold=best_t, opt_balanced_acc=best_bacc,
                    records=records)

---

## Execution

The cell below runs the full pipeline in sequence:

1. **ETL** — scan `../data/` recursively, extract event-locked CWT epochs from all HC and MCI saccade CSVs
2. **XAI** — generate and save the SPM-style group contrast map *before* training, providing a model-agnostic validation of the feature space
3. **Training** — subject-level stratified split → weighted CE loss → 20 epochs with cosine LR → save best checkpoint
4. **Inference demo** — load best checkpoint, run soft-voting ensemble on the first CSV found in the data directory

> **Hardware note:** With `device="cuda"`, training uses FP16 mixed precision on the Jetson AGX Orin's Tensor Cores. On CPU, the code falls back to FP32 automatically with AMP disabled.

In [ ]:
if __name__ == "__main__":
    DATA_DIR = Path("../data")

    pipeline_config = {
        "pre_stimulus_sec":  0.2,
        "post_stimulus_sec": 0.8,
        "min_freq":          1.0,
        "max_freq":          40.0,
        "freq_bins":         40,
        "target_time_bins":  100,
        "w_morlet":          5.0,
        "artifact_threshold": 30.0,   # degrees — epochs with larger error are rejected
    }
    pipeline = EventLockedCWTPipeline(**pipeline_config)

    if not DATA_DIR.exists():
        print(f"[!] 데이터 경로가 존재하지 않습니다: {DATA_DIR}")
    else:
        print(f"[*] 데이터 로드 중: {DATA_DIR.resolve()}")
        pipeline.process_directory(DATA_DIR)

        if not pipeline.data_store:
            print("[!] 처리된 데이터가 없습니다. 경로 및 파일명을 확인하세요.")
        else:
            # --- XAI: Difference Maps (before training) ---
            visualizer = XAIVisualizer(pipeline)
            visualizer.plot_difference_map(pipeline.data_store, save_path="../xai_difference_map.png")

            # --- Dataset ---
            dataset = VOG_CWT_Dataset(pipeline.data_store)
            n_hc  = (dataset.y == 0).sum().item()
            n_mci = (dataset.y == 1).sum().item()
            print(f"[*] 총 {len(dataset)}개 CWT 샘플 (HC_csv_24_25: {n_hc}, MCI: {n_mci})")

            # --- Single-split Training (quick baseline + saves best checkpoint) ---
            model   = EdgeCWTClassifier(num_classes=2, in_channels=4)
            trainer = ModelTrainer(model)
            trainer.train_model(dataset, epochs=50, batch_size=32, patience=10)

            # --- LOSO-CV (rigorous evaluation) ---
            loso = LOSOCrossValidator(dataset, epochs=50, batch_size=32)
            loso_results = loso.run()

            # --- Inference example ---
            engine = JetsonInferenceEngine('best_edge_cwt_model.pth', pipeline_config)
            sample_files = list(DATA_DIR.rglob('*.csv'))
            if sample_files:
                engine.infer_csv(sample_files[0])